# Document Question Answering System (RAG)


---
**Objective:** Build a Retrieval-Augmented Generation (RAG) system that answers questions based on custom documents. The system retrieves relevant information from a document and uses a language model to generate answers grounded in that information.

**Pipeline:**
```
Document → Chunking → Embeddings → FAISS Vector Store → Query → Retrieve → Generate Answer
```

**Tools used (all free, no API key needed):**
- `sentence-transformers` — for creating embeddings
- `FAISS` — local vector database for similarity search
- `flan-t5-base` — open-source LLM for answer generation
- `PyPDF2` — for reading PDF files
- `langchain` — for pipeline orchestration


## Theory — Week 7 Concepts

---

### What is RAG (Retrieval-Augmented Generation)?

LLMs are trained on general data up to a cutoff date — they don't know about your private documents, company data, or recent events. RAG solves this:

1. **Retrieval** — find the most relevant chunks of text from your documents using vector similarity search
2. **Augmentation** — inject those chunks into the LLM's prompt as context
3. **Generation** — the LLM generates an answer using both its training knowledge AND the retrieved context

```
Without RAG: User question → LLM → Answer (might hallucinate)
With RAG:    User question → Retrieve from docs → LLM (question + context) → Grounded answer
```

**Why RAG works:** Instead of the LLM guessing, it now has the actual document text in front of it.

---

### Advanced RAG
Basic RAG has limitations — sometimes the retrieved chunks aren't the most relevant ones. Advanced RAG techniques improve this:

| Technique | What it does |
|---|---|
| **Hybrid Search** | Combines keyword (BM25) + vector search for better retrieval |
| **Re-ranking** | After retrieval, re-score chunks using a cross-encoder for better ordering |
| **Query expansion** | Rephrase or expand the user query to improve retrieval |
| **Chunking strategies** | Sentence-level, semantic, or hierarchical chunking instead of fixed size |
| **Self-RAG** | Model decides when to retrieve and evaluates its own output |
| **HyDE** | Generate a hypothetical answer first, then use that to search |

---

### LLM Internals
How a Large Language Model works:

- **Tokenization** — text is split into tokens (subwords), each mapped to an integer
- **Embedding layer** — each token ID is mapped to a dense vector (embedding)
- **Transformer blocks** — N layers of self-attention + feed-forward networks
- **Self-attention** — each token looks at all other tokens to build context-aware representations
- **Output projection** — final hidden states are projected to vocabulary size → softmax → probabilities
- **Decoding** — sample or take argmax to pick next token

**Key insight:** The attention mechanism is what allows LLMs to understand context. "bank" in "river bank" vs "bank account" gets different representations because attention weights differ.

---

### Fine-Tuning — LoRA / PEFT

Training a full LLM (billions of parameters) from scratch is expensive. Fine-tuning on custom data is cheaper but still requires storing full gradient updates.

**PEFT (Parameter-Efficient Fine-Tuning)** — only update a small fraction of parameters:

| Method | Idea |
|---|---|
| **LoRA (Low-Rank Adaptation)** | Freeze original weights, add small trainable low-rank matrices (A × B) alongside each weight. Only A and B are trained. |
| **Prefix Tuning** | Add trainable tokens to the beginning of each layer's input |
| **Prompt Tuning** | Only add trainable tokens at the input layer |
| **Adapter** | Add small trainable bottleneck layers between existing layers |

**LoRA in detail:**
```
Original: W (d×d) — frozen
LoRA adds: W + ΔW = W + (A × B)   where A is d×r, B is r×d, r << d
Only A and B are trained — reduces trainable params by 10-1000x
```

---

### LLM Evaluation

How do we measure if an LLM or RAG system is good?

| Metric | What it measures |
|---|---|
| **BLEU** | N-gram overlap between generated and reference text |
| **ROUGE** | Recall-oriented overlap (common for summarization) |
| **BERTScore** | Semantic similarity using BERT embeddings |
| **Faithfulness** | Does the answer stay true to the retrieved context? (no hallucination) |
| **Answer Relevance** | Is the answer actually relevant to the question? |
| **Context Recall** | Did retrieval find all the relevant chunks? |
| **Context Precision** | How many retrieved chunks were actually useful? |

**RAGAS** is a popular framework specifically for evaluating RAG pipelines on these dimensions.

---

### ReAct Framework

ReAct = **Re**asoning + **Act**ing — a prompting strategy where the LLM alternates between:
1. **Thought** — reason about what to do next
2. **Action** — call a tool (search, calculator, etc.)
3. **Observation** — receive the tool's output
4. **Repeat** until final answer

```
Question: What is the population of India in 2024?
Thought: I need to search for India's current population.
Action: Search("India population 2024")
Observation: India's population in 2024 is approximately 1.44 billion.
Thought: I now have the answer.
Answer: India's population in 2024 is approximately 1.44 billion.
```

ReAct agents are much more reliable than single-pass LLMs because they can use tools, verify information, and self-correct.

---


## Step 1 — Install Dependencies

All tools are free and open-source — no API keys needed.

In [ ]:
!pip install -q \
    sentence-transformers \
    faiss-cpu \
    PyPDF2 \
    langchain \
    langchain-community \
    transformers \
    torch \
    accelerate

print("All dependencies installed ✅")

## Step 2 — Imports

In [ ]:
import os
import re
import textwrap
import PyPDF2
import numpy as np
import torch

from sentence_transformers import SentenceTransformer
import faiss
from transformers import T5ForConditionalGeneration, T5Tokenizer
import warnings
warnings.filterwarnings("ignore")

print("Imports done ✅")
print("GPU available:", torch.cuda.is_available())

## Step 3 — Load Your Document

You can use **any PDF or text file**. Options:
- Upload your own PDF (notes, resume, research paper)
- Use the built-in sample text below (about Deep Learning & AI — works without any upload)

**To upload your own file:** Uncomment the upload cell and run it.


In [ ]:
# ── Option A: Upload your own PDF ──────────────────────────────────────────
# from google.colab import files
# uploaded = files.upload()
# UPLOADED_FILE = list(uploaded.keys())[0]
# print("Uploaded:", UPLOADED_FILE)

# ── Option B: Use built-in sample text (no upload needed) ──────────────────
# This sample corpus covers Deep Learning, RAG, and LLMs — good for demo

SAMPLE_TEXT = """
Introduction to Deep Learning and Neural Networks

Deep learning is a subfield of machine learning that uses neural networks with many layers
to learn representations from raw data. Unlike traditional machine learning, which requires
manual feature engineering, deep learning models automatically discover the features needed
for classification or detection directly from raw data.

A neural network consists of an input layer, one or more hidden layers, and an output layer.
Each layer contains neurons that apply a weighted sum and an activation function to their inputs.
Training involves adjusting weights using backpropagation and gradient descent to minimize loss.

Convolutional Neural Networks (CNN) are specialized for processing grid-like data such as images.
They use convolutional layers with shared weights to detect spatial features like edges and textures.
Pooling layers reduce spatial dimensions while retaining important information.
CNNs are widely used in image classification, object detection, and medical imaging.

Recurrent Neural Networks (RNN) process sequential data by maintaining a hidden state that
carries information from previous timesteps. However, vanilla RNNs suffer from vanishing gradients,
making it hard to learn long-term dependencies.

LSTM (Long Short-Term Memory) solves the vanishing gradient problem using gates: the forget gate
decides what to discard, the input gate decides what new information to store, and the output gate
determines the next hidden state. LSTM is used for text generation, machine translation, and speech.

GRU (Gated Recurrent Unit) is a simplified version of LSTM with only two gates: update and reset.
It is faster to train and often achieves similar performance to LSTM.

Transformers replaced RNNs for most NLP tasks. The self-attention mechanism allows each token
to attend to all other tokens in the sequence simultaneously, enabling parallel processing.
BERT uses bidirectional transformers for understanding tasks. GPT uses decoder-only transformers
for text generation. Both are pre-trained on massive text corpora and fine-tuned for specific tasks.

Retrieval-Augmented Generation (RAG) combines retrieval and generation. A document is split into
chunks, each converted to an embedding vector. When a user asks a question, the most similar
chunks are retrieved and provided as context to the language model, which generates a grounded answer.
RAG reduces hallucination by grounding the model in actual document content.

Autoencoders are neural networks trained to reconstruct their input through a compressed
bottleneck representation. They are used for dimensionality reduction, denoising, and anomaly detection.
Variational Autoencoders (VAE) learn a probability distribution in latent space, enabling generation.

Generative Adversarial Networks (GAN) consist of a generator that creates fake samples and a
discriminator that tries to distinguish real from fake. They are trained adversarially until the
generator produces realistic outputs. GANs are used for image synthesis and data augmentation.

Transfer Learning allows a model pre-trained on a large dataset to be adapted for a new task
with much less data and compute. Fine-tuning updates only some layers while freezing others.
LoRA (Low-Rank Adaptation) is a parameter-efficient fine-tuning method that adds small trainable
matrices to frozen pre-trained weights, reducing the number of trainable parameters significantly.
"""

print("Sample text loaded ✅")
print(f"Total characters: {len(SAMPLE_TEXT)}")

## Step 4 — Load & Preprocess Document

This function supports both PDF and plain text. If you uploaded a PDF, set `SOURCE = 'pdf'` and update the filename.

In [ ]:
def load_text_from_pdf(pdf_path):
    """Extract text from a PDF file."""
    text = ""
    with open(pdf_path, 'rb') as f:
        reader = PyPDF2.PdfReader(f)
        for page_num, page in enumerate(reader.pages):
            page_text = page.extract_text()
            if page_text:
                text += f"\n[Page {page_num+1}]\n{page_text}"
    return text

# ── Choose source ────────────────────────────────────────────────────────────
SOURCE = 'sample'  # Change to 'pdf' if you uploaded a file

if SOURCE == 'pdf':
    PDF_PATH = 'your_document.pdf'   # replace with your uploaded filename
    raw_text = load_text_from_pdf(PDF_PATH)
    print(f"PDF loaded: {len(raw_text)} characters from {PDF_PATH}")
else:
    raw_text = SAMPLE_TEXT
    print(f"Using sample text: {len(raw_text)} characters")

# Preview
print("\n--- First 300 characters ---")
print(raw_text[:300])

## Step 5 — Text Chunking

We split the document into smaller overlapping chunks.

**Why chunking?**
- LLMs have a context window limit — can't fit a whole book
- Smaller chunks = more precise retrieval (retrieve exactly the relevant part)
- Overlap ensures sentences at chunk boundaries aren't lost

**Parameters:**
- `chunk_size = 300` — characters per chunk
- `chunk_overlap = 50` — overlap between consecutive chunks


In [ ]:
def chunk_text(text, chunk_size=300, chunk_overlap=50):
    """Split text into overlapping chunks."""
    # Clean up whitespace
    text = re.sub(r'\n+', '\n', text).strip()

    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        if chunk.strip():
            chunks.append(chunk.strip())
        start += chunk_size - chunk_overlap  # move forward with overlap

    return chunks

chunks = chunk_text(raw_text, chunk_size=300, chunk_overlap=50)

print(f"Total chunks created : {len(chunks)}")
print(f"Chunk size           : 300 characters")
print(f"Chunk overlap        : 50 characters")
print()
print("--- Sample Chunk 1 ---")
print(chunks[0])
print()
print("--- Sample Chunk 2 ---")
print(chunks[1])

## Step 6 — Create Embeddings

We convert each chunk into a dense vector (embedding) using a pre-trained sentence transformer model.

**What is an embedding?**
A vector of numbers that captures the semantic meaning of a piece of text. Texts with similar meaning have vectors that are close together in high-dimensional space.

**Model used:** `all-MiniLM-L6-v2` — fast, lightweight, 384-dimensional embeddings. Runs locally, completely free.


In [ ]:
print("Loading embedding model...")
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model loaded ✅")

print("\nCreating embeddings for all chunks...")
chunk_embeddings = embed_model.encode(chunks, show_progress_bar=True, convert_to_numpy=True)

print(f"\nEmbedding matrix shape: {chunk_embeddings.shape}")
print(f"Each chunk → {chunk_embeddings.shape[1]}-dimensional vector")

## Step 7 — Build FAISS Vector Store

FAISS (Facebook AI Similarity Search) is a local, fast vector database. We store all chunk embeddings in it so we can do similarity search at query time.

**How similarity search works:**
1. Convert query to embedding vector
2. Compute cosine/dot product similarity between query vector and all stored vectors
3. Return top-K most similar chunks


In [ ]:
# Normalize embeddings for cosine similarity
faiss.normalize_L2(chunk_embeddings)

# Build FAISS index
embedding_dim = chunk_embeddings.shape[1]   # 384
index = faiss.IndexFlatIP(embedding_dim)    # Inner product = cosine similarity (after normalization)
index.add(chunk_embeddings)

print(f"FAISS index built ✅")
print(f"Total vectors stored: {index.ntotal}")
print(f"Embedding dimension : {embedding_dim}")

## Step 8 — Retrieval Function

Given a user query, this function:
1. Embeds the query using the same sentence-transformer model
2. Searches the FAISS index for the most similar chunks
3. Returns the top-K relevant chunks as context


In [ ]:
def retrieve(query, top_k=3):
    """Retrieve the top_k most relevant chunks for a given query."""
    # Embed the query
    query_embedding = embed_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_embedding)

    # Search FAISS index
    scores, indices = index.search(query_embedding, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx != -1:
            results.append({
                'chunk': chunks[idx],
                'score': float(score),
                'chunk_id': int(idx)
            })
    return results

# Test retrieval
test_query = "What is deep learning?"
retrieved = retrieve(test_query, top_k=3)

print(f"Query: '{test_query}'")
print(f"\nTop {len(retrieved)} retrieved chunks:")
for i, r in enumerate(retrieved):
    print(f"\n[Chunk {r['chunk_id']} | Similarity: {r['score']:.4f}]")
    print(r['chunk'])
    print("-" * 60)

## Step 9 — Load Language Model (flan-t5-base)

We use Google's `flan-t5-base` — a free, open-source text-to-text model that runs locally on Colab.

**Why flan-t5?**
- Completely free, no API key
- Good at following instructions and answering questions
- Small enough to run on Colab CPU/GPU
- Text-to-text format: input is a prompt, output is the answer


In [ ]:
print("Loading flan-t5-base model (this may take a minute)...")

tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")
llm_model  = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")

device = "cuda" if torch.cuda.is_available() else "cpu"
llm_model = llm_model.to(device)

print(f"Model loaded ✅ (running on {device})")

## Step 10 — Answer Generation

The RAG pipeline:
1. Retrieve relevant chunks from vector store
2. Build a prompt: `question + retrieved context`
3. Pass to flan-t5 to generate answer


In [ ]:
def generate_answer(question, context_chunks, max_new_tokens=200):
    """Generate an answer using retrieved context."""
    # Build context string from retrieved chunks
    context = "\n\n".join([f"Context {i+1}: {c['chunk']}" for i, c in enumerate(context_chunks)])

    # Build prompt
    prompt = f"""Answer the question based only on the context provided below.
If the answer is not in the context, say "I don't have enough information to answer this."

Context:
{context}

Question: {question}

Answer:"""

    # Tokenize
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=1024,
        truncation=True
    ).to(device)

    # Generate
    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            early_stopping=True,
            temperature=0.7,
            do_sample=False
        )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer

# Test
test_q = "What is deep learning?"
ctx = retrieve(test_q, top_k=3)
ans = generate_answer(test_q, ctx)
print(f"Q: {test_q}")
print(f"A: {ans}")

## Step 11 — Complete RAG Pipeline

Combining everything into one clean `rag_answer()` function.

In [ ]:
def rag_answer(question, top_k=3, verbose=True):
    """
    Full RAG pipeline:
    1. Retrieve relevant chunks
    2. Generate answer using context
    3. Return answer + sources
    """
    # Step 1: Retrieve
    retrieved_chunks = retrieve(question, top_k=top_k)

    # Step 2: Generate
    answer = generate_answer(question, retrieved_chunks)

    if verbose:
        print("=" * 70)
        print(f"QUESTION: {question}")
        print("=" * 70)
        print(f"\nANSWER: {answer}")
        print("\n--- Retrieved Context (Sources) ---")
        for i, chunk in enumerate(retrieved_chunks):
            print(f"\n[Source {i+1} | Similarity: {chunk['score']:.4f}]")
            print(textwrap.fill(chunk['chunk'], width=80))
        print("=" * 70)

    return {
        "question": question,
        "answer": answer,
        "sources": retrieved_chunks
    }

# Test the full pipeline
result = rag_answer("What is an LSTM and how does it work?")

## Step 12 — Ask Multiple Questions

In [ ]:
questions = [
    "What is the difference between LSTM and GRU?",
    "How does the Transformer model work?",
    "What is RAG and why is it useful?",
    "What is LoRA fine-tuning?",
    "What are CNNs used for?",
]

print("Running RAG pipeline on multiple questions...\n")
for q in questions:
    result = rag_answer(q, top_k=3, verbose=False)
    print(f"Q: {q}")
    print(f"A: {result['answer']}")
    print(f"   (Top source similarity: {result['sources'][0]['score']:.4f})")
    print()

## Step 13 — Basic Evaluation

We evaluate the retrieval quality by checking if the retrieved chunks actually contain keywords from the expected answer.

In [ ]:
def evaluate_retrieval(question, expected_keywords, top_k=3):
    """Check if retrieval finds chunks containing expected keywords."""
    retrieved = retrieve(question, top_k=top_k)
    combined_context = " ".join([r['chunk'].lower() for r in retrieved])

    found = [kw for kw in expected_keywords if kw.lower() in combined_context]
    precision = len(found) / len(expected_keywords)

    return {
        "question": question,
        "keywords_found": found,
        "keywords_missing": [kw for kw in expected_keywords if kw not in found],
        "retrieval_precision": precision,
        "top_similarity": retrieved[0]['score'] if retrieved else 0
    }

# Evaluate on known questions
eval_cases = [
    ("What is deep learning?",        ["neural networks", "machine learning", "layers"]),
    ("How does LSTM work?",            ["gates", "forget", "hidden state"]),
    ("What is a GAN?",                 ["generator", "discriminator", "adversarial"]),
    ("What is transfer learning?",     ["pre-trained", "fine-tuning", "dataset"]),
]

import pandas as pd
eval_results = []
for q, keywords in eval_cases:
    r = evaluate_retrieval(q, keywords)
    eval_results.append({
        "Question": q,
        "Keywords Found": len(r['keywords_found']),
        "Total Keywords": len(keywords),
        "Retrieval Precision": f"{r['retrieval_precision']:.2f}",
        "Top Similarity": f"{r['top_similarity']:.4f}"
    })

eval_df = pd.DataFrame(eval_results)
eval_df

## Step 14 — Visualize Retrieval Similarity

In [ ]:
import matplotlib.pyplot as plt

test_questions = [
    "What is deep learning?",
    "How does LSTM work?",
    "What is transformer architecture?",
    "What is RAG system?",
    "What is LoRA fine-tuning?",
]

fig, axes = plt.subplots(1, len(test_questions), figsize=(18, 5))

for ax, q in zip(axes, test_questions):
    results = retrieve(q, top_k=5)
    scores  = [r['score'] for r in results]
    labels  = [f"Chunk\n{r['chunk_id']}" for r in results]

    bars = ax.bar(labels, scores, color=['#2ecc71' if s > 0.5 else '#3498db' for s in scores])
    ax.set_title(q[:30] + "...", fontsize=8, fontweight='bold')
    ax.set_ylabel("Cosine Similarity")
    ax.set_ylim(0, 1)
    ax.tick_params(axis='x', labelsize=7)

    for bar, score in zip(bars, scores):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.02,
                f"{score:.2f}", ha='center', fontsize=7)

plt.suptitle("Retrieval Similarity Scores (Top 5 chunks per query)", fontweight='bold')
plt.tight_layout()
plt.show()

## Step 15 — RAG System Architecture Summary

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════╗
║           RAG SYSTEM ARCHITECTURE — SUMMARY                 ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  INDEXING PIPELINE (run once)                                ║
║  ─────────────────────────────                               ║
║  [Document PDF/TXT]                                          ║
║       ↓                                                      ║
║  [Text Extraction - PyPDF2]                                  ║
║       ↓                                                      ║
║  [Chunking - 300 chars, 50 overlap]                          ║
║       ↓                                                      ║
║  [Embedding - all-MiniLM-L6-v2 → 384-dim vectors]           ║
║       ↓                                                      ║
║  [FAISS Vector Store - IndexFlatIP]                          ║
║                                                              ║
║  QUERY PIPELINE (run per question)                           ║
║  ────────────────────────────────                            ║
║  [User Question]                                             ║
║       ↓                                                      ║
║  [Query Embedding - same model]                              ║
║       ↓                                                      ║
║  [FAISS Similarity Search → Top-K chunks]                    ║
║       ↓                                                      ║
║  [Prompt = Question + Retrieved Context]                     ║
║       ↓                                                      ║
║  [flan-t5-base → Generated Answer]                           ║
║       ↓                                                      ║
║  [Answer + Source Chunks returned to User]                   ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
""")

## Conclusion

**What we built:** A complete end-to-end RAG (Retrieval-Augmented Generation) system for Document Question Answering.

**Pipeline stages implemented:**
- Document Loading (PDF or text)
- Text Chunking (overlapping, 300-char chunks)
- Embedding Creation (sentence-transformers, 384-dim)
- Vector Store (FAISS, cosine similarity)
- Query Processing (embed → search)
- Context Retrieval (top-K similar chunks)
- Answer Generation (flan-t5-base with retrieved context)
- Evaluation (retrieval precision)

**Key learnings:**
- RAG reduces hallucination by grounding LLM responses in actual document content
- Chunking strategy affects retrieval quality — smaller chunks = precise retrieval, larger = more context
- The embedding model determines how well semantic similarity is captured
- FAISS enables fast similarity search even over thousands of chunks
- The LLM's answer is only as good as the retrieved context

**Improvements possible:**
- Use better embedding models (e.g., `bge-large-en`) for higher retrieval accuracy
- Add hybrid search (BM25 + vector) for better recall
- Use a larger/better LLM (e.g., Gemini API free tier) for better generation
- Add re-ranking after retrieval for better chunk ordering
- Implement ReAct-style agent for multi-hop questions

**Concepts applied from Week 7:**
- RAG pipeline (retrieval + augmentation + generation)
- LLM Internals (tokenization, embeddings, transformer)
- Fine-Tuning / LoRA (theory — how to adapt pre-trained models)
- LLM Evaluation (retrieval precision, similarity scores)
- ReAct Framework (theory — reasoning + acting with tools)
